# 04-07 - Data Leakage Prevention

**Phase:** 04 - Data Analysis & Preparation

**Difficulty:** 3/3 | **Priority:** 1/2

**Status:** VERIFIED

---

## 1. What Are We Solving?

**Data leakage** happens when information from outside the training set (e.g. the test set or the future) leaks into training. This gives falsely optimistic results that fail in production. It's one of the most dangerous and common ML mistakes.

## 2. Why Does This Matter?

A model with leakage looks great in evaluation but fails in the real world. Recognizing and preventing leakage is critical for building models that actually work.

## 3. Prerequisites

- Unit 04.5 (Feature scaling & encoding)
- Unit 04.6 (Train/val/test splits)

## 4. Learning Objectives

By the end of this notebook, you should be able to:
- [ ] Define data leakage
- [ ] Recognize scaling-before-splitting leakage
- [ ] Recognize target leakage
- [ ] Recognize temporal leakage
- [ ] Prevent leakage with proper pipelines
## 5. Mental Model

**Mental Model:** Data leakage is like studying for an exam by looking at the answer key. You'll ace the practice test, but you haven't actually learned anything. When you deploy your model, it will fail spectacularly because it memorized the answers instead of learning the patterns. Data leakage happens when information from the future or target accidentally gets into your training process.

Key: understand the data before you model it.


## 2a. Decision Guidance

| Situation | What to Do | Why |
|-----------|------------|-----|
| Scaling before split | Split first, then fit scaler on train only | Prevents test data leakage |
| Encoding before split | Split first, then fit encoder on train only | Prevents test data leakage |
| Feature engineering uses target | Use target encoding with cross-validation | Prevents target leakage |
| Time series with future features | Only use lag features up to current time | Prevents temporal leakage |
| Feature selection uses all data | Select features using only training data | Prevents information leakage |


## 2b. Common Mistakes to Avoid

- Fitting preprocessing on entire dataset before splitting
- Using target encoding without cross-validation
- Including time-derived features that use future information
- Performing feature selection on all data before splitting
- Not checking for proxy variables that leak the target


## 6. Preprocessing Leakage: Scaling Before Splitting

If you fit a scaler on the whole dataset (including test), the test data influences the scaling, leaking information.


In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

np.random.seed(42)
n = 1000
X = np.random.normal(0, 1, (n, 5))
y = (X[:, 0] + X[:, 1] > 0).astype(int)

# LEAKAGE: scale the whole dataset, then split
scaler = StandardScaler()
X_scaled_all = scaler.fit_transform(X)
X_tr, X_te, y_tr, y_te = train_test_split(X_scaled_all, y, test_size=0.2, random_state=1)
model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
leak_acc = accuracy_score(y_te, model.predict(X_te))

# CORRECT: split first, then fit scaler on train only
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=1)
scaler = StandardScaler().fit(X_tr)
X_tr_s = scaler.transform(X_tr)
X_te_s = scaler.transform(X_te)
model = LogisticRegression(max_iter=1000).fit(X_tr_s, y_tr)
correct_acc = accuracy_score(y_te, model.predict(X_te_s))

print(f"Accuracy with leakage: {leak_acc:.3f}")
print(f"Accuracy without leakage: {correct_acc:.3f}")
print("\nThe leakage version uses test information in scaling, which is wrong.")
print("Always split BEFORE fitting any scaler/imputer.")


Accuracy with leakage: 0.980
Accuracy without leakage: 0.980

The leakage version uses test information in scaling, which is wrong.
Always split BEFORE fitting any scaler/imputer.


## 7. Target Leakage

**Target leakage** happens when a feature contains information about the target that wouldn't be available at prediction time. Example: predicting whether a patient has a disease using a feature that's only known after diagnosis.


In [2]:
# Target leakage: a feature that reveals the target
np.random.seed(2)
n = 500
# True signal
disease = np.random.choice([0, 1], n, p=[0.8, 0.2])

# Legitimate features (weak signal)
age = np.random.randint(20, 80, n)
bp = 100 + 0.5 * age + np.random.normal(0, 10, n)

# LEAKED feature: 'test_result' only known after diagnosis (leaks the target)
test_result = disease + np.random.normal(0, 0.1, n)

X_clean = np.column_stack([age, bp])
X_leaked = np.column_stack([age, bp, test_result])

X_tr, X_te, y_tr, y_te = train_test_split(X_clean, disease, test_size=0.3, random_state=2)
model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
clean_acc = accuracy_score(y_te, model.predict(X_te))

X_tr, X_te, y_tr, y_te = train_test_split(X_leaked, disease, test_size=0.3, random_state=2)
model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
leaked_acc = accuracy_score(y_te, model.predict(X_te))

print(f"Accuracy without leaked feature: {clean_acc:.3f}")
print(f"Accuracy WITH leaked feature: {leaked_acc:.3f}")
print("\nThe leaked feature gives near-perfect accuracy, but it's not available")
print("at prediction time in the real world. The model would fail in production.")


Accuracy without leaked feature: 0.773
Accuracy WITH leaked feature: 1.000

The leaked feature gives near-perfect accuracy, but it's not available
at prediction time in the real world. The model would fail in production.


## 8. Temporal Leakage

Using future data to predict the past. Example: predicting next month's sales using this month's actual sales (which you don't have yet).


In [3]:
# Temporal leakage: using future data
np.random.seed(3)
t = np.arange(200)
sales = 100 + 0.5 * t + np.random.normal(0, 5, 200)

# LEAKED feature: next month's actual sales (not available at prediction time)
next_sales = np.roll(sales, -1)
next_sales[-1] = sales[-1]

X_leak = next_sales[:-1].reshape(-1, 1)
y = sales[1:]

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
model = LinearRegression().fit(X_leak, y)
mse = mean_squared_error(y, model.predict(X_leak))

print(f"MSE using next month's actual sales: {mse:.3f}")
print("\nThis looks great, but you can't know next month's sales when predicting it!")
print("This is temporal leakage - the model is useless in practice.")


MSE using next month's actual sales: 0.000

This looks great, but you can't know next month's sales when predicting it!
This is temporal leakage - the model is useless in practice.


## 9. Leakage in Imputation

Fitting an imputer on the whole dataset (including test) leaks test information into the imputation statistics.


In [4]:
# Leakage in imputation
from sklearn.impute import SimpleImputer

np.random.seed(4)
n = 500
X = np.random.normal(50, 10, (n, 2))
y = (X[:, 0] > 50).astype(int)

# Introduce missing values
X_missing = X.copy()
X_missing[np.random.choice(n, 50, replace=False), 0] = np.nan

# LEAKAGE: fit imputer on all data
imp = SimpleImputer(strategy="mean").fit(X_missing)
X_imp_all = imp.transform(X_missing)
X_tr, X_te, y_tr, y_te = train_test_split(X_imp_all, y, test_size=0.2, random_state=4)
model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
leak_acc = accuracy_score(y_te, model.predict(X_te))

# CORRECT: split first, fit imputer on train only
X_tr, X_te, y_tr, y_te = train_test_split(X_missing, y, test_size=0.2, random_state=4)
imp = SimpleImputer(strategy="mean").fit(X_tr)
X_tr_i = imp.transform(X_tr)
X_te_i = imp.transform(X_te)
model = LogisticRegression(max_iter=1000).fit(X_tr_i, y_tr)
correct_acc = accuracy_score(y_te, model.predict(X_te_i))

print(f"Accuracy with imputation leakage: {leak_acc:.3f}")
print(f"Accuracy without leakage: {correct_acc:.3f}")
print("\nFit the imputer on training data only.")


Accuracy with imputation leakage: 0.940
Accuracy without leakage: 0.950

Fit the imputer on training data only.


## 10. Preventing Leakage with Pipelines

scikit-learn **pipelines** chain preprocessing and modeling, ensuring transformations are fit on training folds only during cross-validation.


In [5]:
# Use a pipeline to prevent leakage
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score

np.random.seed(5)
X = np.random.normal(0, 1, (500, 3))
y = (X[:, 0] > 0).astype(int)

# Pipeline: scale then model - scaler fit on each training fold only
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
scores = cross_val_score(pipe, X, y, cv=5)

print(f"Pipeline CV accuracy: {scores.mean():.3f} +/- {scores.std():.3f}")
print("\nThe pipeline ensures the scaler is fit on training folds only,")
print("preventing leakage during cross-validation.")


Pipeline CV accuracy: 0.986 +/- 0.019

The pipeline ensures the scaler is fit on training folds only,
preventing leakage during cross-validation.


## 11. Failure Case: Real-World Leakage

A common real-world leak: including a feature that's a direct function of the target, or using data collected after the prediction time.


In [6]:
# Real-world leak: feature is a function of the target
np.random.seed(6)
n = 400
credit_score = np.random.normal(650, 50, n)
defaulted = (credit_score < 600).astype(int)  # target

# Leaked feature: 'risk_label' assigned by a human AFTER knowing the outcome
risk_label = defaulted + np.random.normal(0, 0.05, n)

X_leak = np.column_stack([credit_score, risk_label])
X_clean = credit_score.reshape(-1, 1)

X_tr, X_te, y_tr, y_te = train_test_split(X_leak, defaulted, test_size=0.3, random_state=6)
model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
leak_acc = accuracy_score(y_te, model.predict(X_te))

X_tr, X_te, y_tr, y_te = train_test_split(X_clean, defaulted, test_size=0.3, random_state=6)
model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
clean_acc = accuracy_score(y_te, model.predict(X_te))

print(f"Accuracy with leaked 'risk_label': {leak_acc:.3f}")
print(f"Accuracy with only credit_score: {clean_acc:.3f}")
print("\nThe 'risk_label' leaks the outcome. In production, you wouldn't have it")
print("before predicting default - so the model is useless.")


Accuracy with leaked 'risk_label': 1.000
Accuracy with only credit_score: 1.000

The 'risk_label' leaks the outcome. In production, you wouldn't have it
before predicting default - so the model is useless.


## 12. Debugging: Common Errors

- **Scaling/imputing before splitting**: preprocessing leakage.
- **Features that reveal the target**: target leakage.
- **Future data in training**: temporal leakage.
- **Tuning on the test set**: evaluation leakage.
- **Not using pipelines**: accidental leakage in CV.

## 13. Real-World Considerations

- Always split before any preprocessing.
- Question every feature: would I have this at prediction time?
- Use pipelines to keep preprocessing consistent.
- Be suspicious of suspiciously good results.

## 14. Common Mistakes

- Fitting scalers on all data.
- Including post-outcome features.
- Random splits on time series.
- Tuning on the test set.

## 15. When NOT to Use

- Don't include features unavailable at prediction time.
- Don't fit preprocessing on test data.
- Don't use future data for past predictions.

## 16. Challenge

Identify the leakage in this scenario: predicting customer churn using a 'satisfaction_survey' feature that's only collected from customers who stayed. Explain why it leaks.


In [7]:
# Challenge: identify leakage
np.random.seed(7)
n = 300
churned = np.random.choice([0, 1], n, p=[0.7, 0.3])

# Leaked feature: survey only collected from non-churned customers
survey = np.where(churned == 0, np.random.normal(8, 1, n), np.nan)

print("Survey availability by churn status:")
print(f"  Non-churned (0): {np.sum(~np.isnan(survey[churned==0]))} have survey")
print(f"  Churned (1): {np.sum(~np.isnan(survey[churned==1]))} have survey")
print("\nThe survey is only collected from customers who stayed (non-churned).")
print("So 'has a survey' perfectly predicts 'did not churn' - this is target leakage.")
print("At prediction time, you don't know if a customer will churn, so you can't")
print("know whether they'd have a survey. The feature leaks the target.")


Survey availability by churn status:
  Non-churned (0): 210 have survey
  Churned (1): 0 have survey

The survey is only collected from customers who stayed (non-churned).
So 'has a survey' perfectly predicts 'did not churn' - this is target leakage.
At prediction time, you don't know if a customer will churn, so you can't
know whether they'd have a survey. The feature leaks the target.


## 19. Knowledge Check

1. What is data leakage and why is it dangerous?
2. What is the difference between data leakage and overfitting?
3. How does preprocessing before train/test split cause leakage?
4. What is temporal leakage in time series data?
5. How do you detect data leakage in an existing pipeline?

## 18. Teach-Back Questions

Explain to another person:

- Why scaling before splitting is leakage.
- How a feature can leak the target.
- Why suspiciously good results are a red flag.

## 19. Summary

You now understand data leakage: preprocessing leakage, target leakage, and temporal leakage. You know how to prevent it by splitting before preprocessing, questioning features, and using pipelines. This is critical for building models that work in production.


## 21a. Exit Criteria

- [ ] I can identify all forms of data leakage
- [ ] I can prevent leakage in preprocessing pipelines
- [ ] I can implement target encoding with cross-validation
- [ ] I can audit a pipeline for temporal leakage
- [ ] I can explain why leakage causes inflated model performance

## 21b. Next Step

Proceed to `04_08_synthesis.ipynb` to apply everything you learned in a comprehensive exercise.

## 22. Hands-On Practice

**Level 1 - Observation:** Examine the code in this notebook and identify where leakage could occur.

**Level 2 - Guided:** Fix a leaking pipeline by moving the scaler inside the cross-validation loop.

**Level 3 - Practice:** Implement proper target encoding with cross-validation.

**Level 4 - Challenge:** Find and fix data leakage in a provided buggy pipeline.

**Level 5 - Mastery:** Audit a complete ML pipeline for all forms of data leakage and document findings.

## 21. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy, pandas, scikit-learn
OUTPUTS: PASS
LAST VERIFIED: 2026-08-28
```
